# 01 - Exploratory Data Analysis

Statistical analysis required by the course: descriptive statistics, distribution shape, normality testing, Pearson vs. Spearman correlation, the Central Limit Theorem in action, and bias detection.

Run `python src/data_collection.py && python src/preprocessing.py` first so `data/processed/combined_data.csv` exists.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "combined_data.csv", index_col=0, parse_dates=True)
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df.head()

## Descriptive statistics

In [ ]:
print(df.describe())
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df["demand"], bins=50, edgecolor="black")
axes[0, 0].set_title("Electricity Demand Distribution")
axes[0, 0].set_xlabel("Demand (MW)")

df.groupby("hour")["demand"].mean().plot(ax=axes[0, 1])
axes[0, 1].set_title("Average Demand by Hour of Day")
axes[0, 1].set_xlabel("Hour")
axes[0, 1].set_ylabel("Avg Demand (MW)")

df.groupby("dayofweek")["demand"].mean().plot(ax=axes[1, 0])
axes[1, 0].set_title("Average Demand by Day of Week (0=Mon)")
axes[1, 0].set_xlabel("Day of week")

sample_month = df.loc[df.index.to_period("M").astype(str) == df.index.to_period("M").astype(str).iloc[0]]
sample_month["demand"].plot(ax=axes[1, 1])
axes[1, 1].set_title("Demand Time Series (first month in data)")
axes[1, 1].set_ylabel("Demand (MW)")

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "01_distribution_analysis.png", dpi=200)
plt.show()

## Normality tests (course requirement)

Shapiro-Wilk on a sample of hourly demand. Electricity demand has a well-known bimodal/multimodal shape (weekday-vs-weekend, day-vs-night), so we expect to reject normality.

In [ ]:
sample = df["demand"].sample(min(5000, len(df)), random_state=42)
stat, p_value = stats.shapiro(sample)
print(f"Shapiro-Wilk statistic={stat:.4f}, p-value={p_value:.4e}")
print("Normal" if p_value > 0.05 else "NOT normal -> use non-parametric stats / non-linear models")

## Correlation analysis (course requirement): Pearson vs. Spearman

Pearson assumes roughly linear relationships between (approximately) normal variables. Since demand is non-Gaussian and its relationship with weather is not perfectly linear (e.g. heating/cooling both increase demand as temperature moves away from a comfort point), Spearman's rank correlation is a useful cross-check.

In [ ]:
numeric_cols = ["demand", "temperature", "humidity", "precipitation", "cloud_cover", "wind_speed"]
pearson_corr = df[numeric_cols].corr(method="pearson")
spearman_corr = df[numeric_cols].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(pearson_corr, annot=True, cmap="coolwarm", center=0, square=True, ax=axes[0])
axes[0].set_title("Pearson Correlation")
sns.heatmap(spearman_corr, annot=True, cmap="coolwarm", center=0, square=True, ax=axes[1])
axes[1].set_title("Spearman Correlation")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "02_correlation.png", dpi=200)
plt.show()

print(f"demand vs temperature (Pearson):  {pearson_corr.loc['demand', 'temperature']:.3f}")
print(f"demand vs temperature (Spearman): {spearman_corr.loc['demand', 'temperature']:.3f}")

## Central Limit Theorem demonstration (course requirement)

Raw hourly demand is non-Gaussian. Aggregating to daily averages should move the distribution closer to Gaussian, per the CLT.

In [ ]:
daily_demand = df["demand"].resample("D").mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["demand"], bins=50, edgecolor="black")
stat_h, p_h = stats.shapiro(df["demand"].sample(min(5000, len(df)), random_state=42))
axes[0].set_title(f"Hourly Demand (Shapiro p={p_h:.2e})")

axes[1].hist(daily_demand, bins=50, edgecolor="black", color="seagreen")
stat_d, p_d = stats.shapiro(daily_demand)
axes[1].set_title(f"Daily-Mean Demand (Shapiro p={p_d:.2e})")

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "03_clt_demonstration.png", dpi=200)
plt.show()

## Bias detection (course requirement)

In [ ]:
missing_pct = df.isnull().mean() * 100
print("Missing data %:")
print(missing_pct[missing_pct > 0] if missing_pct.any() else "none")

print(f"\nTemporal coverage: {df.index.min()} to {df.index.max()} ({df.index.year.nunique()} year(s))")

weekend_demand = df.loc[df.index.dayofweek >= 5, "demand"].mean()
weekday_demand = df.loc[df.index.dayofweek < 5, "demand"].mean()
print(f"\nWeekday mean demand: {weekday_demand:.0f} MW")
print(f"Weekend mean demand: {weekend_demand:.0f} MW")
print(f"Relative difference: {(weekday_demand - weekend_demand) / weekday_demand * 100:.1f}%")
print("-> Confirms strong weekday/weekend seasonality the model must capture (is_weekend, dayofweek features).")